In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Generación Aumentada por Recuperación (RAG) Multimodal usando la API de Gemini en Vertex AI

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/retrieval-augmented-generation/intro_multimodal_rag.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Ejecutar en Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fuse-cases%2Fretrieval-augmented-generation%2Fintro_multimodal_rag.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Ejecutar en Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/retrieval-augmented-generation/intro_multimodal_rag.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo"><br> Ver en GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/use-cases/retrieval-augmented-generation/intro_multimodal_rag.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Abrir en Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://goo.gle/40z3Fun">
        <img width="32px" src="https://cdn.qwiklabs.com/assets/gcp_cloud-e3a77215f0b8bfa9b3f611c0d2208c7e8708ed31.svg" alt="Google Cloud logo"><br> Abrir en Skills
    </a>
  </td>
</table>

<div style="clear: both;"></div>


## Descripción general

La generación aumentada por recuperación (RAG) se ha convertido en un paradigma popular para permitir que los LLM accedan a datos externos y también como un mecanismo de grounding para mitigar las alucinaciones.

En este notebook, aprenderás cómo realizar RAG multimodal, donde realizarás Q&A sobre un documento financiero lleno tanto de texto como de imágenes.

### Comparando RAG basado en texto y RAG multimodal

El RAG multimodal ofrece varias ventajas sobre el RAG basado en texto:

1. **Acceso a conocimiento enriquecido:** El RAG multimodal puede acceder y procesar tanto información textual como visual, proporcionando una base de conocimientos más rica y completa para el LLM.
2. **Capacidades de razonamiento mejoradas:** Al incorporar pistas visuales, el RAG multimodal puede realizar inferencias mejor informadas a través de diferentes tipos de modalidades de datos.

Este notebook te muestra cómo usar RAG con la API de Gemini en Vertex AI, [embeddings de texto](https://cloud.google.com/vertex-ai/docs/generative-ai/model-reference/text-embeddings) y [embeddings multimodales](https://cloud.google.com/vertex-ai/docs/generative-ai/model-reference/multimodal-embeddings), para construir un motor de búsqueda de documentos.

### Objetivos

Este notebook proporciona una guía para construir un motor de búsqueda de documentos utilizando generación aumentada por recuperación (RAG) multimodal, paso a paso:

1. Extraer y almacenar metadatos de documentos que contienen tanto texto como imágenes, y generar embeddings para los documentos.
2. Buscar en los metadatos con consultas de texto para encontrar texto o imágenes similares.
3. Buscar en los metadatos con consultas de imagen para encontrar imágenes similares.
4. Usando una consulta de texto como entrada, buscar respuestas contextuales utilizando tanto texto como imágenes.

## Primeros pasos

In [ ]:
%pip install --upgrade --quiet google-genai
%pip install --quiet pymupdf

In [ ]:
from google import genai

PROJECT_ID = "tu-proyecto-id"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [ ]:
from IPython.display import Markdown, display
from google.genai.types import GenerateContentConfig, Part

In [ ]:
MODEL_ID = "gemini-2.5-flash"

## Construcción de metadatos de documentos que contienen texto e imágenes

### Importar funciones auxiliares para construir metadatos

Antes de construir el sistema RAG multimodal, es importante tener metadatos de todo el texto e imágenes del documento. Para fines de referencias y citas, los metadatos deben contener elementos esenciales, incluyendo número de página, nombre de archivo, contador de imágenes, etc. Por lo tanto, como siguiente paso, generarás embeddings a partir de los metadatos, lo cual es necesario para realizar búsquedas por similitud al consultar los datos.

In [ ]:
# Importar la función get_document_metadata de las utilidades
# (Asegúrate de tener el archivo de utilidades en tu entorno)
from intro_multimodal_rag_utils import get_document_metadata

### Extraer y almacenar metadatos de texto e imágenes de un documento

In [ ]:
# Especificar la carpeta con los PDFs
pdf_folder_path = "data/"

# Especificar el prompt para la descripción de imágenes
image_description_prompt = """Explica qué está sucediendo en la imagen.
Si es una tabla, extrae todos los elementos de la tabla.
Si es un gráfico, explica los hallazgos del gráfico.
No incluyas números que no estén mencionados en la imagen.
"""

# Extraer metadatos de texto e imagen del documento PDF
text_metadata_df, image_metadata_df = get_document_metadata(
    client,
    pdf_folder_path,
    image_save_dir="images",
    image_description_prompt=image_description_prompt,
    embedding_size=1408,
)

print("\n\n --- Procesamiento completado. ---")